In [92]:
import torch
import torch.nn as nn

import torchvision
from torchvision import transforms as T
from torch.utils.data import Dataset, DataLoader
# import albumentations as A

import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

from pathlib import Path
import os

In [93]:
train_dir = Path("dataset/dataset/train")
val_dir = Path("dataset/dataset/val")

In [94]:
def get_classes(directory):
    classes_ls = sorted([item.name for item in directory.iterdir() if not item.name.startswith(".")])
    classes_dict = {j: i for (i, j) in enumerate(classes_ls)}

    return classes_ls, classes_dict

In [95]:
get_classes(train_dir)

(['no', 'sphere', 'vort'], {'no': 0, 'sphere': 1, 'vort': 2})

In [96]:
class LensingDataset(Dataset):
    def __init__(self, train_dir, transform=None):
        self.train_dir = train_dir
        self.transform = transform

        self.images = list(self.train_dir.glob("*/*.npy"))
        self.classes_ls, self.classes_dict = get_classes(train_dir)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        img_path = self.images[index]
        img_class = self.classes_dict[img_path.parent.stem]
        image = np.load(img_path)[0].astype(np.float32)

        if self.transform is not None:
            image = self.transform(image)

        return image, img_class

In [97]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [98]:
train_transform = T.Compose([
    T.ToTensor(),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomVerticalFlip(p=0.5)
])

val_transform = T.Compose([
    T.ToTensor(),
])

In [99]:
train_data = LensingDataset(train_dir=train_dir, transform=train_transform)
val_data = LensingDataset(train_dir=val_dir, transform=val_transform)

In [100]:
train_dataloader = DataLoader(
    dataset=train_data,
    batch_size=32,
    shuffle=True
)

val_dataloader = DataLoader(
    dataset=val_data,
    batch_size=32,
    shuffle=False
)

In [101]:
train_data[0]

(tensor([[[0.0117, 0.0156, 0.0091,  ..., 0.0182, 0.0091, 0.0130],
          [0.0156, 0.0104, 0.0130,  ..., 0.0195, 0.0169, 0.0169],
          [0.0078, 0.0143, 0.0130,  ..., 0.0169, 0.0091, 0.0091],
          ...,
          [0.0104, 0.0182, 0.0091,  ..., 0.0065, 0.0156, 0.0182],
          [0.0078, 0.0091, 0.0065,  ..., 0.0117, 0.0091, 0.0195],
          [0.0104, 0.0143, 0.0091,  ..., 0.0104, 0.0143, 0.0208]]]),
 0)

In [102]:
next(iter(train_dataloader))

[tensor([[[[0.0090, 0.0141, 0.0090,  ..., 0.0141, 0.0230, 0.0077],
           [0.0051, 0.0090, 0.0102,  ..., 0.0141, 0.0128, 0.0192],
           [0.0102, 0.0128, 0.0141,  ..., 0.0051, 0.0115, 0.0064],
           ...,
           [0.0141, 0.0128, 0.0141,  ..., 0.0090, 0.0205, 0.0102],
           [0.0102, 0.0038, 0.0115,  ..., 0.0166, 0.0115, 0.0154],
           [0.0166, 0.0141, 0.0064,  ..., 0.0051, 0.0090, 0.0115]]],
 
 
         [[[0.0129, 0.0153, 0.0188,  ..., 0.0153, 0.0118, 0.0094],
           [0.0106, 0.0129, 0.0106,  ..., 0.0094, 0.0082, 0.0118],
           [0.0071, 0.0071, 0.0047,  ..., 0.0141, 0.0129, 0.0071],
           ...,
           [0.0094, 0.0118, 0.0047,  ..., 0.0059, 0.0153, 0.0094],
           [0.0118, 0.0047, 0.0129,  ..., 0.0059, 0.0106, 0.0106],
           [0.0059, 0.0129, 0.0082,  ..., 0.0071, 0.0141, 0.0141]]],
 
 
         [[[0.0063, 0.0100, 0.0038,  ..., 0.0188, 0.0125, 0.0125],
           [0.0100, 0.0063, 0.0075,  ..., 0.0088, 0.0100, 0.0088],
           [0.0038

In [103]:
class TinyVGG(nn.Module):
    def __init__(self, in_channels, hidden_units, num_classes):
        super().__init__()

        self.conv_block_1 = nn.Sequential(
            nn.Conv2d(
                in_channels=in_channels,
                out_channels=hidden_units,
                kernel_size=3,
                padding=1,
                stride=1,
                bias=False
            ),
            nn.BatchNorm2d(hidden_units),
            nn.ReLU(),
        
            nn.Conv2d(
                in_channels=hidden_units,
                out_channels=hidden_units,
                kernel_size=3,
                padding=1,
                stride=1,
                bias=False
            ),
            nn.BatchNorm2d(hidden_units),
            nn.ReLU(),

            nn.MaxPool2d(kernel_size=2),
        )

        self.conv_block_2 = nn.Sequential(
            nn.Conv2d(
                in_channels=hidden_units,
                out_channels=hidden_units,
                kernel_size=3,
                padding=1,
                stride=1,
                bias=False
            ),
            nn.BatchNorm2d(hidden_units),
            nn.ReLU(),

            nn.Conv2d(
                in_channels=hidden_units,
                out_channels=hidden_units,
                kernel_size=3,
                padding=1,
                stride=1,
                bias=False
            ),
            nn.BatchNorm2d(hidden_units),
            nn.ReLU(),

            nn.MaxPool2d(kernel_size=2),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_features=hidden_units * 37 * 37,
                      out_features=num_classes)
        )

    def forward(self, x):
        x = self.conv_block_1(x)
        x = self.conv_block_2(x)
        return self.classifier(x)

In [104]:
torch.manual_seed(42)
torch.cuda.manual_seed(42)

model0 = TinyVGG(1, 64, len(train_data.classes_ls)).to(device)

In [105]:
rand_batch = torch.randn((32, 1, 150, 150)).to(device)

model0.eval()
with torch.inference_mode():
    output = model0(rand_batch)

output.shape, rand_batch.shape

(torch.Size([32, 3]), torch.Size([32, 1, 150, 150]))